## Homework

In this homework, we'll deploy the Straight vs Curly Hair Type model we trained in the
[previous homework](../08-deep-learning/homework.md).

Download the model files from here:

* https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data
* https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx

With wget:

```bash
PREFIX="https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
DATA_URL="${PREFIX}/hair_classifier_v1.onnx.data"
MODEL_URL="${PREFIX}/hair_classifier_v1.onnx"
wget ${DATA_URL}
wget ${MODEL_URL}
```


## Question 1

To be able to use this model, we need to know the name of the input and output nodes.

What's the name of the output:

* `output`
* `sigmoid`
* `softmax`
* `prediction`


In [4]:
import onnx

MODEL_PATH = "hair_classifier_v1.onnx"  # or hair_classifier_empty.onnx if you want


model = onnx.load(MODEL_PATH)
graph = model.graph

print("== Inputs ==")
for i, inp in enumerate(graph.input):
    shape = []
    for d in inp.type.tensor_type.shape.dim:
        if d.dim_param:
            shape.append(d.dim_param)
        else:
            shape.append(d.dim_value)
    print(f"{i}: name={inp.name}, shape={shape}")

print("\n== Outputs ==")
for i, out in enumerate(graph.output):
    shape = []
    for d in out.type.tensor_type.shape.dim:
        if d.dim_param:
            shape.append(d.dim_param)
        else:
            shape.append(d.dim_value)
    print(f"{i}: name={out.name}, shape={shape}")




== Inputs ==
0: name=input, shape=['s77', 3, 200, 200]

== Outputs ==
0: name=output, shape=['s77', 1]



## Preparing the image

You'll need some code for downloading and resizing images. You can use
this code:



In [6]:
from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

## Question 2: Target size

Let's download and resize this image:

https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

Based on the previous homework, what should be the target size for the image?

* 64x64
* 128x128
* 200x200
* 256x256

-> 200x200



## Question 3

Now we need to turn the image into numpy array and pre-process it.

> Tip: Check the previous homework. What was the pre-processing
> we did there?

After the pre-processing, what's the value in the first pixel, the R channel?

* -10.73
* -1.073
* 1.073
* 10.73


In [10]:
import numpy as np
from PIL import Image
from urllib import request
from io import BytesIO
def preprocess(img):
    img = img.convert("RGB")
    img = img.resize((200, 200), Image.NEAREST)

    # raw image → numpy array → float32 in [0, 1]
    x = np.array(img).astype("float32") / 255.0   # (H, W, 3)

    # normalization (same as HW8)
    x = (x - mean[None, None, :]) / std[None, None, :]

    # convert HWC → CHW
    x = np.transpose(x, (2, 0, 1))  # (3, 200, 200)
    return x
# ==========
# CONFIG
# ==========

URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


img = download_image(URL)
x = preprocess(img)

first_pixel_R = x[0, 0, 0]  # channel 0 = R
print("First pixel R after preprocessing:", first_pixel_R)
x.shape

First pixel R after preprocessing: -1.073294


(3, 200, 200)


## Question 4

Now let's apply this model to this image. What's the output of the model?

* 0.09
* 0.49
* 0.69
* 0.89

In [16]:
from io import BytesIO
from urllib import request

import numpy as np
from PIL import Image
import onnxruntime as ort

MODEL_PATH = "hair_classifier_v1.onnx"  # for Q2–Q4
TARGET_SIZE = (200, 200)  # <-- put here the size you used in HW8
IMG_MEAN = np.array([0.485, 0.456, 0.406], dtype="float32")
IMG_STD = np.array([0.229, 0.224, 0.225], dtype="float32")
def preprocess(img):
    img = img.convert("RGB")
    img = img.resize((200, 200), Image.NEAREST)

    x = np.array(img).astype("float32") / 255.0
    x = (x - IMG_MEAN[None, None, :]) / IMG_STD[None, None, :]
    x = np.transpose(x, (2, 0, 1))  # (3, H, W)

    return np.expand_dims(x, axis=0)  # (1, 3, 200, 200)

class HairClassifierONNX:
    def __init__(self, model_path: str = MODEL_PATH):
        self.session = ort.InferenceSession(
            model_path,
            providers=["CPUExecutionProvider"],
        )
        self.input_name = self.session.get_inputs()[0].name
        self.output_name = self.session.get_outputs()[0].name

    def predict_proba_from_array(self, x: np.ndarray) -> float:
        """
        x: np.ndarray of shape (1, 3, H, W)
        returns a scalar probability (float)
        """
        preds = self.session.run([self.output_name], {self.input_name: x})[0]
        # usually (1, 1) or (1, 2) – adjust if your model is different
        return float(preds.squeeze())

    def predict_from_url(self, url: str) -> float:
        img = download_image(url)
        img = prepare_image(img, TARGET_SIZE)
        x = preprocess(img)               # (1, 3, 200, 200) returned from preprocess()
        proba = self.predict_proba_from_array(x)
        return proba



URL = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
clf = HairClassifierONNX(MODEL_PATH)
proba = clf.predict_from_url(URL)
print("Model output (probability):", proba)


Model output (probability): 0.09156641364097595



## Prepare the lambda code

Now you need to copy all the code into a separate python file. You will
need to use this file for the next two questions.

Tip: you can test this file locally with `ipython` or Jupyter Notebook
by importing the file and invoking the function from this file.


## Docker

For the next two questions, we'll use a Docker image that we already
prepared. This is the Dockerfile that we used for creating the image:

```docker
FROM public.ecr.aws/lambda/python:3.13

COPY hair_classifier_empty.onnx.data .
COPY hair_classifier_empty.onnx .
```

Note that it uses Python 3.13.

The docker image is published to [`agrigorev/model-2025-hairstyle:v1`](https://hub.docker.com/r/agrigorev/model-2025-hairstyle).

A few notes:

* The image already contains a model and it's not the same model
  as the one we used for questions 1-4.



## Question 5

Download the base image `agrigorev/model-2025-hairstyle:v1`. You can do it with [`docker pull`](https://docs.docker.com/engine/reference/commandline/pull/).

So what's the size of this base image?

* 88 Mb
* 208 Mb
* 608 Mb
* 1208 Mb

You can get this information when running `docker images` - it'll be in the "SIZE" column.


In [ ]:
# docker pull agrigorev/model-2025-hairstyle:v1
#
# v1: Pulling from agrigorev/model-2025-hairstyle
# b71754c34aa3: Pull complete
# d2379533db7f: Pull complete
# 26c6d8a1e1c2: Pull complete
# 94f2ecca3b37: Pull complete
# f9e394d707b7: Pull complete
# 15b946295de2: Pull complete
# 55d9a27bb275: Pull complete
# cd9c39ee4ab8: Pull complete
# Digest: sha256:9e43d5a5323f7f07688c0765d3c0137af66d0154af37833ed721d6b4de6df528
# Status: Downloaded newer image for agrigorev/model-2025-hairstyle:v1
# docker.io/agrigorev/model-2025-hairstyle:v1

# docker images
# REPOSITORY                                 TAG       IMAGE ID       CREATED       SIZE
# agrigorev/model-2025-hairstyle             v1        4528ad1525d5   7 days ago    608MB


## Question 6

Now let's extend this docker image, install all the required libraries
and add the code for lambda.

You don't need to include the model in the image. It's already included.
The name of the file with the model is `hair_classifier_empty.onnx` and it's
in the current workdir in the image (see the Dockerfile above for the
reference).
The provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.

Now run the container locally.

Score this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

What's the output from the model?

* -1.0
* -0.10
* 0.10
* 1.0

In [ ]:
# docker run -p 9000:8080 hair-lambda
# curl -X POST "http://localhost:9000/2015-03-31/functions/function/invocations" \
#     -d '{"url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'
#
# {"statusCode": 200, "body": "{\"score\": -0.10220836102962494}"}(venv_p3p13_datatalk)


## Publishing it to AWS

Now you can deploy your model to AWS!

* Publish your image to ECR
* Create a lambda function in AWS, use the ECR image
* Give it more RAM and increase the timeout
* Test it
* Expose the lambda function using API Gateway

This is optional and not graded.


## Submit the results

* Submit your results here: https://courses.datatalks.club/ml-zoomcamp-2025/homework/hw09
* If your answer doesn't match options exactly, select the closest one. If the answer is exactly in between two options, select the higher value.

## Publishing to Docker hub

Just for the reference, this is how we published our image to Docker hub:

```bash
docker build -t model-2025-hairstyle -f homework.dockerfile .
docker tag model-2025-hairstyle:latest agrigorev/model-2025-hairstyle:v1
docker push agrigorev/model-2025-hairstyle:v1
```

(You don't need to execute this code)